# RAG(Retrieval Augmented Generation)
- [RAG](https://python.langchain.com/v0.1/docs/modules/data_connection/)은 *Retrieval Augmented Generation*의 약자로, **검색 기반 생성 기법**을 의미한다. 이 기법은 LLM이 특정 문서에 기반하여 보다 정확하고 신뢰할 수 있는 답변을 생성할 수 있도록 돕는다.     
- 사용자의 질문에 대해 자체적으로 구축한 데이터베이스(DB)나 외부 데이터베이스에서 질문과 관련된 문서를 검색하고, 이를 질문과 함께 LLM에 전달한다.
- LLM은 같이 전달된 문서를 바탕으로 질문에 대한 답변을 생성한다. 
- 이를 통해 LLM이 학습하지 않은 내용도 다룰 수 있으며, 잘못된 정보를 생성하는 환각 현상(*hallucination*)을 줄일 수 있다.

## RAG와 파인튜닝(Fine Tuning) 비교

### 파인튜닝(Fine Tuning)

- **정의**: 사전 학습(pre-trained)된 LLM에 특정 도메인의 데이터를 추가로 학습시켜 해당 도메인에 특화된 맞춤형 모델로 만드는 방식이다.
- **장점**
  - 특정 도메인에 최적화되어 높은 정확도와 성능을 낼 수 있다.
- **단점**
  - 모델 재학습에 많은 시간과 자원이 필요하다.
  - 새로운 정보가 반영되지 않으며, 이를 위해서는 다시 학습해야 한다.

### RAG

- **정의**: 모델을 다시 학습시키지 않고, 외부 지식 기반에서 정보를 검색하여 실시간으로 답변에 활용하는 방식이다.
- **장점**
  - 최신 정보를 쉽게 반영할 수 있다.
  - 모델을 수정하지 않아도 되므로 효율적이다.
- **단점**
  - 검색된 문서의 품질에 따라 답변의 정확성이 달라질 수 있다.
  - 검색 시스템 구축이 필요하다.

## 정리

| 항목       | 파인튜닝 | RAG |
| -------- | ---- | --- |
| 도메인 최적화  | 가능   | 제한적 |
| 최신 정보 반영 | 불가능  | 가능  |
| 구현 난이도   | 높음   | 보통  |
| 유연성      | 낮음   | 높음  |

- LLM은 학습 당시의 데이터만을 기반으로 작동하므로 최신 정보나 기업 내부 자료와 같은 특정한 지식 기반에 접근할 수 없다.
- 파인튜닝은 시간과 비용이 많이 들고 유지보수가 어렵다.
-	반면, RAG는 기존 LLM을 변경하지 않고도 외부 문서를 통해 그 한계를 보완할 수 있다.
- RAG는 특히 빠르게 변화하는 정보를 다루는 분야(예: 기술 지원, 뉴스, 법률 등)에서 유용하게 활용된다. 반면, 정적인 정보에 대해 높은 정확도가 필요한 경우에는 파인튜닝이 효과적이다.


## RAG 작동 단계
- 크게 "**정보 저장(인덱싱)**", "**검색**, **생성**"의 단계로 나눌 수 있다.
  
### 1. 정보 저장(인덱싱)
RAG는 사전에 정보를 가공하여 **벡터 데이터베이스**(Vector 저장소)에 저장해 두고, 나중에 검색할 수 있도록 준비한다. 이 단계는 다음과 같은 과정으로 이루어진다.

1. **Load (불러오기)**
   - 답변시 참조할 사전 정보를 가진 데이터들을 불러온다.
2. **Split/Chunking (문서 분할)**
   - 긴 텍스트를 일정한 길이의 작은 덩어리(*chunk*)로 나눈다.
   - 이렇게 해야 검색과 생성의 정확도를 높일 수 있다.
3. **Embedding (임베딩)**
   - 각 텍스트 조각을 **임베딩 벡터**로 변환한다.
   - 임베딩 벡터는 그 문서의 의미를 벡터화 한 것으로 질문과 유사한 문서를 찾을 때 인덱스로 사용된다.
4. **Store (저장)**
   - 임베딩된 벡터를 **벡터 데이터베이스**(벡터 저장소)에 저장한다.
   - 벡터 데이터베이스는 유사한 질문이나 문장을 빠르게 찾을 수 있도록 특화된 데이터 저장소이다.
   
![rag](figures/rag1.png)

### 2. 검색, 생성

사용자가 질문을 하면 다음과 같은 절차로 답변이 생성된다.
1. **Retrieve (검색)**
   - 사용자의 질문을 임베딩한 후, 이 질문 벡터와 유사한 context 벡터를 벡터 데이터베이스에서 검색하여 찾는다.
2. **Query (질의 생성)**
   - 벡터 데이터베이스에서 검색된 문서 조각과 사용자의 질문을 함께 **프롬프트**(prompt)로 구성하여 LLM에 전달한다.
3. **Generation (응답 생성)**
   - LLM은 받은 프롬프트에 대한 응답을 생성한다.
   
- **RAG 흐름**
  
![Retrieve and Generation](figures/rag2.png)


# Document Loader
- LLM에게 질의할 때 같이 제공할 Data들을 저장하기 위해 먼저 읽어들인다.(Load)
- 데이터 Resouce는 다양하다.
    - 데이터를 로드(load)하는 방식은 저장된 위치와 형식에 따라 다양하다. 
      - 로컬 컴퓨터(Local Computer)에 저장된 문서
        - 예: CSV, Excel, JSON, TXT 파일 등
      - 데이터베이스(Database)에 저장된 데이터셋
      - 인터넷에 존재하는 데이터
        - 예: 웹에 공개된 API, 웹 페이지에 있는 데이터, 클라우드 스토리지에 저장된 파일 등

![rag_load](figures/rag_load.png)

- 다양한 문서 형식(format)에 맞춰 읽어오는 다양한 **document loader** 들을 Langchain에서 지원한다.
    - 다양한 Resource들로 부터 데이터를 읽기 위해서는 다양한 라이브러리를 이용해 서로 다른 방법으로 읽어야 한다.
    - Langchain은 데이터를 읽는 다양한 방식의 코드를 하나의 interface로 사용 할 수 있도록 지원한다.
    - 다양한 3rd party library(ppt, github 등등 다양한 3rd party lib도 있음. )들과 연동해 다양한 Resource로 부터 데이터를 Loading 할 수 있다.
        - https://python.langchain.com/docs/integrations/document_loaders/
- **모든 document loader는 기본적으로 동일한 interface(사용법)로 호출할 수있다.**
- **반환타입**
    - **list[Document]**
    - Load 한 문서는 Document객체에 정보들을 넣는다. 여러 문서를 읽을 수 있기 대문에 list에 묶어서 반환한다.
        - **Document 속성**
            - page_content: 문서의 내용
            - metadata(option): 문서에 대한 메타데이터(정보)를 dict 형태로 저장한다. 
            - id(option): 문서의 고유 id
     
- **주의**
    - Langchain을 이용해 RAG를 구현할 때 **꼭 Langchain의 DocumentLoader를 사용해야 하는 것은 아니다.**
    - DocumentLoader는 데이터를 읽어오는 것을 도와주는 라이브러리일 뿐이다. 다른 라이브러리를 이용해서 읽어 들여도 상관없다. 

## 주요 Document Loader

### Text file
- TextLoader 이용

In [1]:
! uv pip install langchain-community -U

Resolved 47 packages in 524ms
Checked 47 packages in 12ms


In [2]:
from langchain_community.document_loaders import TextLoader
path = "data/olympic.txt"

C:\Users\Playdata\AppData\Local\Temp\ipykernel_33264\2883918916.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
with open(path, "rt", encoding="utf-8") as f:
    print(f.read()[:100])

올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하


In [4]:
loader = TextLoader(path, encoding="utf-8")
docs = loader.load() # 실행시 바로 읽는다. -> list[Document]
# docs = loader.lazy_load() # 읽은 문서를 사용할 때 읽는다. -> generator[Document]
print(type(docs), len(docs))
print(type(docs[0]))
# for doc in docs:
#     print(type(doc))

<class 'list'> 1
<class 'langchain_core.documents.base.Document'>


In [5]:
doc = docs[0]
print("문서정보: doc.metadata")
print(doc.metadata)
# 필요한 정보를 추가할 수 있다. LLM에 전달할 프롬프트에 전달한 내용과 검색할 때 사용할 내용
doc.metadata["category"] = "sports"
doc.metadata["tag"] = ["올림픽", "IOC", "동계올림픽", "하계올림픽"]
print(doc.metadata)

문서정보: doc.metadata
{'source': 'data/olympic.txt'}
{'source': 'data/olympic.txt', 'category': 'sports', 'tag': ['올림픽', 'IOC', '동계올림픽', '하계올림픽']}


In [6]:
print("문서내용: doc.page_content")
print(doc.page_content[:200])

문서내용: doc.page_content
올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열


In [7]:
print("문서 식별자 -id: doc.id")
print(doc.id)

문서 식별자 -id: doc.id
None


### PDF
- PyPDF, Pymupdf 등 다양한 PDF 문서를 읽어들이는 파이썬의  3rd party library들을 이용해 pdf 문서를 Load 한다.
    - https://python.langchain.com/docs/integrations/document_loaders/#pdfs
- 각 PDF Loader 특징
    -  PyMuPDFLoader
        -   텍스트 뿐 아니라 이미지, 주석등의 정보를 추출하는데 성능이 좋다.
        -   PyMuPDF 라이브러리 기반
    - PyPDFLoader
        - 텍스트를 빠르게 추출 할 수있다.
        - PyPDF2 라이브러리 기반. 경량 라이브러리로 빠르고 큰 파일도 효율적으로 처리한다.
    - PDFPlumberLoader
        - 표와 같은 복잡한 구조의 데이터 처리하는데 강력한 성능을 보여준다. 텍스트, 이미지, 표 등을 모두 추출할 수 있다. 
        - PDFPlumber 라이브러리 기반
- 설치 패키지
    - DocumentLoader와 연동하는 라이브러리들을 설치 해야 한다.
    - `pip install pypdf -qU`
    - `pip install pymupdf -qU`
    - `pip install pdfplumber -qU`

In [8]:
! uv pip install pypdf pymupdf pdfplumber

Checked 3 packages in 97ms


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, PDFPlumberLoader
path = "data/novel/금_따는_콩밭_김유정.pdf"
loader = PyPDFLoader(path, mode="single") # single : 전체를 하나의 문서로 읽기 page : 페이지당 하나의 문서로 반환
docs = loader.load()
print(len(docs))

In [ ]:
from pprint import pprint
doc = docs[0]
pprint(doc.metadata)
doc.metadata["author"] = "김유정"
pprint(doc.metadata)


{'author': 'Unknown',
 'creationdate': '2024-11-24T07:05:35+00:00',
 'creator': 'Wikisource',
 'moddate': '2024-11-24T07:05:37+00:00',
 'producer': 'Wikisource',
 'source': 'data/novel/금_따는_콩밭_김유정.pdf',
 'title': '금 따는 콩밭',
 'total_pages': 23}
{'author': '김유정',
 'creationdate': '2024-11-24T07:05:35+00:00',
 'creator': 'Wikisource',
 'moddate': '2024-11-24T07:05:37+00:00',
 'producer': 'Wikisource',
 'source': 'data/novel/금_따는_콩밭_김유정.pdf',
 'title': '금 따는 콩밭',
 'total_pages': 23}


In [ ]:
print(doc.page_content)

1
금  따는  콩밭
Exported from Wikisource on 2024 년  11 월  24 일
2
위키백과
위키백과에  이  글
과  관련된 
자료가  있습니다 .
금  따는  콩밭
🙝🙟
땅속  저  밑은  늘  음침하
다 .
고달픈  간드렛불 , 맥없이
푸르끼하다 .
밤과  달라서  낮엔  되우  흐릿하였다 .
겉으로  황토  장벽으로  앞뒤좌우가  콕  막힌  좁직한  구뎅이 .
흡사히  무덤  속같이  귀중중하다 . 싸늘한  침묵 , 쿠더브레한
흙내와  징그러운  냉기만이  그  속에  자욱하다 .
곡괭이는  뻔질  흙을  이르집는다 . 암팡스러이  내려쪼며 ,
퍽  퍽  퍼억 .
이렇게  메떨어진  소리뿐 . 그러나  간간  우수수  하고  벽이  헐
린다 .
영식이는  일손을  놓고  소맷자락을  끌어당기어  얼굴의  땀을
훑는다 . 이놈의  줄이  언제나  잡힐는지  기가  찼다 . 흙  한줌을
집어  코밑에  바짝  들여대고  손가락으로  샅샅이  뒤져본다 . 완
연히  버력은  좀  변한  듯싶다 . 그러나  불통버력이  아주  다  풀
린  것도  아니었다 . 밀똥버력이라야  금이  온다는데  왜  이리
안  나오는지 .
곡괭이를  다시  집어든다 . 땅에  무릎을  꿇고  궁뎅이를  번쩍
든  채  식식거린다 . 곡괭이는  무작정  내려찍는다 . 바닥에서
3
물이  스미어  무르팍이  흔건히  젖었다 . 굿엎은  천판에서  흙방
울은  내리며  목덜미로  굴러든다 . 어떤  때에는  웃벽의  한쪽이
떨어지며  등을  탕  때리고  부서진다 .
그러나  그는  눈도  하나  깜짝하지  않는다 . 금을  캔다고  콩밭
하나를  다  잡쳤다 . 약이  올라서  죽을둥  살둥  눈이  뒤집힌  이
판이다 . 손바닥에  침을  탁  뱉고  곡괭이  자루를  한번  꼰아잡
더니  쉴  줄  모른다 .
등뒤에서는  흙  긁는  소리가  드윽드윽  난다 . 아직도  버력을
다  못  친  모양 . 이  자식이  일을  하나  시졸  하나 . 

In [ ]:
loader = PyMuPDFLoader(path)

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, PDFPlumberLoader
path = "data/novel/금_따는_콩밭_김유정.pdf"
loader = PyMuPDFLoader(path)
docs = loader.load()
print(len(docs))

23


In [ ]:
from pprint import pprint
doc = docs[0]
pprint(doc.metadata)

{'author': 'Unknown',
 'creationDate': "D:20241124070535+00'00'",
 'creationdate': '2024-11-24T07:05:35+00:00',
 'creator': 'Wikisource',
 'file_path': 'data/novel/금_따는_콩밭_김유정.pdf',
 'format': 'PDF 1.4',
 'keywords': '',
 'modDate': "D:20241124070537+00'00'",
 'moddate': '2024-11-24T07:05:37+00:00',
 'page': 0,
 'producer': 'Wikisource',
 'source': 'data/novel/금_따는_콩밭_김유정.pdf',
 'subject': '',
 'title': '금 따는 콩밭',
 'total_pages': 23,
 'trapped': ''}


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, PDFPlumberLoader
path = "data/novel/금_따는_콩밭_김유정.pdf"
loader = PDFPlumberLoader(path)
docs = loader.load()
print(len(docs))

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

23


In [ ]:
from pprint import pprint
doc = docs[0]
pprint(doc.metadata)

{'Author': 'Unknown',
 'CreationDate': "D:20241124070535+00'00'",
 'Creator': 'Wikisource',
 'ModDate': "D:20241124070537+00'00'",
 'Producer': 'Wikisource',
 'Title': '금 따는 콩밭',
 'file_path': 'data/novel/금_따는_콩밭_김유정.pdf',
 'page': 0,
 'source': 'data/novel/금_따는_콩밭_김유정.pdf',
 'total_pages': 23}


### Web 문서 로드

#### WebBaseLoader를 이용해 Web 문서로딩

requests와 BeautifulSoup을 이용해 web 페이지의 내용을 크롤링해서 Document로 loading한다.

- 주요 파라미터
  - **web_paths***: str | list[str]
    - 크롤링할 대상 URL
  - **requests_kwargs**: dict
    - requests.get() 에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    - headers, cookies, verify 등 설정 전달
  - **header_template**: dict
    - HTTP Header 에 넣을 값을 dict 로 전달.
  - **encoding**
    - requests의 응답 encoding을 설정 (bs_kwargs의 from_encoding 보다 상위에서 적용됨)
  - **bs_kwargs**
    - BeautifulSoup initializer에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    -  주요 옵션
       - **parse_only**: 요청 페이지에서 특정 요소만 선택해서 가져오기. **SoupStrainer를 사용**한다.
         - BeautifulSoup의 `SoupStrainer` 를 이용해 페이지의 일부분만 가져오기
           - 웹 페이지를 파싱(parse, 구조 분석)할 때, 페이지 전체가 아닌 특정 부분만 필요한 경우가 많다. BeautifulSoup 라이브러리의 SoupStrainer를 사용하면, 원하는 태그나 속성이 있는 요소만 골라서 파싱할 수 있다.
           - BeautifulSoup("html문서", parse_only=Strainer객체)
               - Strainer객체에 지정된 영역에서만 내용 찾는다.
           - `SoupStrainer("태그명")`, `SoupStrainer(["태그명", "태그명"])`
             - 지정한 태그 만 조회
           - `SoupStrainer(name="태그명", attrs={속성명:속성값})`
             -  지정한 태그 중 속성명=속성값인 것만 조회
        - **from_encoding**: Encoding 설정 
          - "from_encoding":"utf-8"
   - **bs_get_text_kwargs**:
     - BeautifulSoup객체.get_text() 에 전달할 파라미터 dict로 전달. (key: parameter변수명, value: 전달할 값)
     - **RAG 구축시 `separator` 와 `strip=True` 으로 설정하는 것이 좋다.** (RAG 품질을 위해 강력히 권장되는 설정이다.)
       -  get_text() 는 기본적으로 태그를 제거하고 텍스트만 이어 붙여 반환한다. `separator=구분자문자` 를 지정하여 추출된 텍스트 요소들 사이에 원하는 구분자를 지정할 수있다. `\n` 을 구분자로 사용하면 텍스트 블록 사이에 줄바꿈이 들어가 **문단의 구조를 어느정도 살릴 수 있다.**
       -  웹 문서의 줄바꿈도 포함해서 읽기 때문에 공백과 줄바꿈이 혼재된 상태로 반환된다. `strip=True`로 설정하면 추출된 문자 앞뒤의 공백 문자들을 제거할 수있다.

In [ ]:
! uv pip install beautifulsoup4 requests
! uv pip install ixml

Checked 2 packages in 30ms
Checked 1 package in 6ms


In [ ]:

from bs4 import BeautifulSoup
html_txt = """<html> \n
<body>  
<p><b>제목</b> <span></dspan>  
<p>다음문단</p>
<div> 다음 내용</div>  
</body>  
</html>  
""" 
soup = BeautifulSoup(html_txt)  
txt1= soup.get_text()
txt2= soup.get_text(strip=True)
txt3 = soup.get_text(strip=True, separator="\n")  
print("---기본---")  
print(txt1)
print("---strip=True---")
print(txt2)
print("---strip=True, separator=\\n---")


---기본---


제목 
다음문단
 다음 내용



---strip=True---
제목다음문단다음 내용
---strip=True, separator=\n---


In [ ]:
from bs4 import SoupStrainer, BeautifulSoup
strainer = SoupStrainer("span")
soup2 = BeautifulSoup(html_txt, parse_only=strainer)
soup2

<span>
<p>다음문단</p>
<div> 다음 내용</div>
</span>

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"


url = [
     "https://m.sports.naver.com/fifaworldcup2026/article/025/0003533131",
    "https://m.sports.naver.com/wbaseball/article/079/0004161711"
    ]
loader = WebBaseLoader(
    web_path=url[1],
    bs_kwargs = {
        "parse_only":SoupStrainer(name="div", attrs={"class":"_article_content"})
    },
    bs_get_text_kwargs={
        "strip":True,
        "separator":"\n\n"
    }
    # default_parser="lxml".
)


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
docs = loader.load()
print(len(docs))
pprint(docs[0].metadata)
print(docs[0].page_content)

1
{'source': 'https://m.sports.naver.com/wbaseball/article/079/0004161711'}
샌프란시스코 이정후. 연합뉴스

메이저 리그(MLB) 한국인 외야수

이정후

(

샌프란시스코

)가 연이틀 멀티 히트(1경기 2안타)를 터뜨렸다.

이정후는 25일(한국 시각) 미국 캘리포니아주 오라클 파크에서 열린

애슬레틱스

와 홈 경기에 6번 타자 우익수로 선발 출전해 4타수 2안타를 기록했다. 전날 애슬레틱스와 홈 경기 3타수 2안타까지 2경기 연속 멀티 히트다.

시즌 타율을 3할3푼1리에서 3할3푼3리로 올렸다. 그러나 이날

오토 로페스

(마이애미)가 3타수 2안타로 타율을 3할4푼으로 올려 이정후와 격차라 오히려 벌어졌다.

이정후는 이날 출발부터 좋았다. 2회 첫 타석에서 애슬레틱스 좌완 선발

게이지 점프

의 5구째 속구를 통타, 우익수 쪽으로 2루타를 날렸다. 시즌 19호 2루타.

5회 직선타로 물러난 이정후는 7회 멀티 히트를 완성했다. 2루수 쪽 깊은 타구로 내야 안타를 만들었다.

샌프란시스코 이정후의 호수비 장면. 연합뉴스

이정후는 수비에서도 빛났다. 9회초 2사 1, 2 위기에서 이정후는 조나 하임의 깊숙한 타구를 펜스에 부딪히며 잡아내는 투혼을 펼쳤다.

결정적인 이정후의 수비는 팀을 깨웠다. 샌프란시스코는 9회말 선두 타자

라파엘 데버스

가 동점 홈런을 때린 데 이어 2사에서 빅터 베리코토가 역시 1점 홈런을 터뜨리며 경기를 극적으로 끝냈다.

이정후는 9회말 마지막 타석에서 우익수 뜬공으로 물러났지만 베리코토의 끝내기포에 기쁨을 만끽했다. 이정후의 수비가 없었다면 샌프란시스코도 승리를 장담하기 어려웠다.


#### RecursiveUrlLoader

- 주어진 URL에서 시작하여 그 페이지 안의 내부 링크를 재귀적으로 따라가며 여러 웹 문서를 자동 수집하여 로드한다.
  - 시작 url을 요청/페이지를 파싱 한 뒤에 `<a href>` 들을 수집하고 그 페이지들을 요청/페이지 파싱을 한다. 
- WebBaseLoader가 단일 페이지(단일 URL) 단위라면 RecursiveUrlLoader는 **웹 사이트 구조 전체를 크롤링하는 전용 수집기**에 가깝다.
  ```bash
  시작 URL
  ├─ 내부 링크 1
  │   ├─ 내부 링크 1-1
  │   └─ 내부 링크 1-2
  ├─ 내부 링크 2
  └─ 내부 링크 3
  ```
위 구조일때 무든 페이지를 재귀적으로 수집한다.
- 주요 파라미터
  - **url**: 시작 url
  - **max_depth**
    - 링크를 몇 단계 **깊이** 까지 따라갈지 제한
    - 사이트 폭주를 막기 위한 안전장치
      - **0**: 시작페이지만, **1**: 시작페이지 + 1차링크, **2**(기본값): 시작페이지 + 1차링크 + 2차링크
  - **exclude_dirs**: list[str]
    - 크롤링 제외 경로
    - ex) `exclude_dirs=['/login', 'signup']`
  - **prevent_outside**: bool
    - True: base_url 바깥 링크는 가져오지 않고 무시한다.
  - **base_url**: str
    - prevent_outside=True일 때 바깥링크의 기준. 없으면 `url`(시작 url)의 host가 된다. 
  - **extractor**
    - 문서 내용 추출 사용자 정의 함수
    - default는 응답 받은 페이지를 `BeautifulSoup(응답페이지).get_text()` 로 텍스트를 추출한다.
    - ````python
        def custom_extractor(html:str) ->str:
            # 웹 페이지 문서를 입력으로 받는다.
            soup = BeautifulSoup(html, 'lxml')
            return soup.select_one('article').get_text() # 원하는 항목을 추출해서 반환한다.
        
        loader = RecursiveUrlLoader(
            url=start_url,
            extractor=custom_extractor
        )    
    ```

In [ ]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
start_URL = "https://docs.python.org/3/"

def extractor(html:str) -> str:
    """
    """
    soup = BeautifulSoup(html)
    body_element = soup.select_one("div.body")
    return body_element.get_text(strip=True, separator="\n") if body_element is not None else soup.get_text(strip=True, separator="\n")


loader = RecursiveUrlLoader(
    url=start_URL,
    max_depth=2, 
    prevent_outside=True,
    base_url=start_URL,
    extractor=extractor
)
docs = loader.load()

C:\Users\Playdata\AppData\Local\Temp\ipykernel_19812\2935226735.py:8: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html)
c:\Users\Playdata\SKN\SKN31\10_AI_Agent\.venv\Lib\site-packages\langchain_community\document_loaders\recursive_url_loader.py:44: XMLParsedAsHTMLWarning: It looks like you're usin

In [ ]:
len(docs)

29

In [ ]:
idx = 1
doc = docs[idx]
doc.metadata
# pprint(doc.metadata)


{'source': 'https://docs.python.org/3/contents.html',
 'content_type': 'text/html',
 'title': 'Python Documentation contents — Python 3.14.6 documentation',
 'description': 'What’s New in Python- What’s new in Python 3.14- Summary – Release highlights, New features- PEP 649& PEP 749: Deferred evaluation of annotations, PEP 734: Multiple interpreters in the standard...',
 'language': 'en'}

In [ ]:
# for doc in docs:
#     print(doc.metadata["source"])


In [ ]:
print(doc.page_content)

Python Documentation contents
¶
What’s New in Python
What’s new in Python 3.14
Summary – Release highlights
New features
PEP 649
&
PEP 749
: Deferred evaluation of annotations
PEP 734
: Multiple interpreters in the standard library
PEP 750
: Template string literals
PEP 768
: Safe external debugger interface
A new type of interpreter
Free-threaded mode improvements
Improved error messages
PEP 784
: Zstandard support in the standard library
Asyncio introspection capabilities
Concurrent safe warnings control
Other language changes
Built-ins
Command line and environment
PEP 758: Allow
except
and
except*
expressions without brackets
PEP 765: Control flow in
finally
blocks
Garbage collection
Default interactive shell
New modules
Improved modules
argparse
ast
asyncio
calendar
concurrent.futures
configparser
contextvars
ctypes
curses
datetime
decimal
difflib
dis
errno
faulthandler
fnmatch
fractions
functools
getopt
getpass
graphlib
heapq
hmac
http
imaplib
inspect
io
json
linecache
logging.han

### <del>ArxivLoader</del>

- arxiv api가 업데이트 되고 ArxivLoader는 그에 맞춰 업데이트가 되지 않아 ArxivLoader는 정상적으로 실행되지 않는다. **arxiv api 를 이용해서 검색 후 pdf를 다운로드 받는다.**
- arxiv API: https://github.com/lukasschwab/arxiv.py
- [arXiv-아카이브](https://arxiv.org/) 는 미국 코렐대학에서 운영하는 **무료 논문 저장소**로, 물리학, 수학, 컴퓨터 과학, 생물학, 금융, 경제 등 **과학, 금융 분야의 논문**들을 공유한다.
- 설치
  - `pip install arxiv`



In [ ]:
# ! uv pip install arxiv

In [ ]:
import arxiv 
search = arxiv.Search(
    query="Advanced RAG",
    max_results=10,
    sort_by=arxiv.SortCriterion.Relevance


)
client = arxiv.Client()
results = client.results(search)
print(type(results))

<class 'itertools.islice'>


In [ ]:
paper = next(results) # for in 대신
print(type(paper))
print(paper.title)
print(paper.authors[0].name)
print(paper.categories)
print(paper.summary)
print(paper.pdf_url)
print(paper.published)
print(paper.get_short_id())

<class 'arxiv.Result'>
MultiHop-RAG: Benchmarking Retrieval-Augmented Generation for Multi-Hop Queries
Yixuan Tang
['cs.CL']
Retrieval-augmented generation (RAG) augments large language models (LLM) by retrieving relevant knowledge, showing promising potential in mitigating LLM hallucinations and enhancing response quality, thereby facilitating the great adoption of LLMs in practice. However, we find that existing RAG systems are inadequate in answering multi-hop queries, which require retrieving and reasoning over multiple pieces of supporting evidence. Furthermore, to our knowledge, no existing RAG benchmarking dataset focuses on multi-hop queries. In this paper, we develop a novel dataset, MultiHop-RAG, which consists of a knowledge base, a large collection of multi-hop queries, their ground-truth answers, and the associated supporting evidence. We detail the procedure of building the dataset, utilizing an English news article dataset as the underlying RAG knowledge base. We demonst

In [ ]:
import os
import requests
save_dir = "data/papers"
os.makedirs(save_dir, exist_ok=True)
resp = requests.get(paper.pdf_url)
if resp.status_code==200:
    with open(os.path.join(save_dir, paper.get_short_id()+".pdf"), "wb") as fo:
        fo.write(resp.content)

In [ ]:
#논문pdf download 함수
import os
import requests

def download_arxiv_paper(paper:arxiv.Result, dirpath:str):
    os.makedirs(dirpath, exist_ok=True)
    resp = requests.get(paper.pdf_url)
    if resp.status_code==200:
        with open(os.path.join(dirpath, paper.get_short_id()+".pdf"), "wb") as fo:
            fo.write(resp.content)

In [ ]:
save_dir = "data/papers"
for paper in results:
    download_arxiv_paper(paper, save_dir)

In [ ]:
# arxiv에서 검색어의 논문을 조회해서 다운 받은 후에
# document에 page_content에는 논문 내용을 METADATA에는 위의 정보를 넣어서
# list[document]를 반환화는 함수

import arxiv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
def load_arxiv_docs(
        query: str,
        top_k: int = 10,
        dirpath: str="." # 현재디렉토리 저장
) -> list[Document]:

    client = arxiv.Client()
    search = arxiv.Search(
        query=query, max_results=top_k, sort_by=arxiv.SortCriterion.Relevance
    )
    results = client.results(search)
    docs = []
    for paper in results:
        download_arxiv_paper(paper, dirpath)
        file_path = os.path.join(dirpath, paper.get_short_id()+".pdf")
        loader = PyPDFLoader(file_path, mode="single")
        doc = loader.load()[0]
        doc.metadata = {
            "title": paper.title,
            "authors": [a.name for a in paper.authors],
            "categories": paper.categories,
            "arxiv_url": paper.entry_id,
            "pdf_url": paper.pdf_url
        }
        docs.append(doc)
    return docs



In [ ]:
docs = load_arxiv_docs(query="transformers", top_k=20, dirpath="data/papers/transformers")

PdfReadError("Invalid Elementary Object starting with b'\\\\' @756840: b'ateVersion (2021.1) \\\\par /Author()/Title()/Subject()/Creator(LaTeX with hyperref'")
Multiple definitions in dictionary at byte 0xb8c76 for key /Author
Multiple definitions in dictionary at byte 0xb8c7e for key /Title


### Docling
- IBM Research에서 개발한 오픈소스 문서처리 도구로 다양한 종류의 문서를 구조화된 데이터로 변환해 생성형 AI에서 활용할 수있도록 지원한다.
- **주요기능**
  - PDF, DOCX, PPTX, XLSX, HTML, 이미지 등 여러 형식을 지원
  - PDF의 **페이지 레이아웃, 읽기 순서, 표 구조, 코드, 수식** 등을 분석하여 정확하게 읽어들인다.
  - OCR을 지원하여 스캔된 PDF나 이미지에서 텍스트를 추출할 수있다.
  - 읽어들인 내용을 markdown, html, json등 다양한 형식으로 출력해준다.
- 설치 : `pip install langchain-docling ipywidgets -qU` 
- 참조
  - docling 사이트: https://github.com/docling-project/docling
  - 랭체인-docling https://python.langchain.com/docs/integrations/document_loaders/docling/

In [ ]:
# ! uv pip install langchain-docling torch transformers accelerate ipywidgets -qU

In [7]:
from huggingface_hub import login
import os
from dotenv import load_dotenv
load_dotenv()

access_key = os.getenv("HUGGINGFACE_API_KEY")
login(access_key)

In [ ]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# 현재 torch와 docling이 사용하는 OCR lib(RApidOCR) 호환성 문제 때문에 OCR 기능을 끄느 설정
pdf_option = PdfPipelineOptions()
pdf_option.do_ocr = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pdf_option
        )
    }
)

path = ["data/papers/1804.09256v2.pdf", "data/papers/2401.15391v1.pdf"]
loader = DoclingLoader(
    file_path=path,
    export_type=ExportType.MARKDOWN,
    converter=converter
)

In [21]:
docs = loader.load()

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Stage preprocess failed for run 1, pages [33]: std::bad_alloc
Stage preprocess failed for run 1, pages [34]: std::bad_alloc
Stage preprocess failed for run 1, pages [35]: std::bad_alloc
Stage preprocess failed for run 1, pages [36]: std::bad_alloc
Stage preprocess failed for run 1, pages [37]: std::bad_alloc


In [25]:
doc =docs[1]
doc.metadata

{'source': 'data/papers/2401.15391v1.pdf'}

In [26]:
print(doc.page_content)

## MultiHop-RAG: Benchmarking Retrieval-Augmented Generation for Multi-Hop Queries

Yixuan Tang and Yi Yang

Hong Kong University of Science and Technology {yixuantang,imyiyang}@ust.hk

## Abstract

Retrieval-augmented generation (RAG) augments large language models (LLM) by retrieving relevant knowledge, showing promising potential in mitigating LLM hallucinations and enhancing response quality, thereby facilitating the great adoption of LLMs in practice. However, we find that existing RAG systems are inadequate in answering multi-hop queries, which require retrieving and reasoning over multiple pieces of supporting evidence. Furthermore, to our knowledge, no existing RAG benchmarking dataset focuses on multihop queries. In this paper, we develop a novel dataset, MultiHop-RAG , which consists of a knowledge base, a large collection of multihop queries, their ground-truth answers, and the associated supporting evidence. We detail the procedure of building the dataset, utilizing an Engl

In [27]:

from IPython.display import Markdown
Markdown(doc.page_content)

## MultiHop-RAG: Benchmarking Retrieval-Augmented Generation for Multi-Hop Queries

Yixuan Tang and Yi Yang

Hong Kong University of Science and Technology {yixuantang,imyiyang}@ust.hk

## Abstract

Retrieval-augmented generation (RAG) augments large language models (LLM) by retrieving relevant knowledge, showing promising potential in mitigating LLM hallucinations and enhancing response quality, thereby facilitating the great adoption of LLMs in practice. However, we find that existing RAG systems are inadequate in answering multi-hop queries, which require retrieving and reasoning over multiple pieces of supporting evidence. Furthermore, to our knowledge, no existing RAG benchmarking dataset focuses on multihop queries. In this paper, we develop a novel dataset, MultiHop-RAG , which consists of a knowledge base, a large collection of multihop queries, their ground-truth answers, and the associated supporting evidence. We detail the procedure of building the dataset, utilizing an English news article dataset as the underlying RAG knowledge base. We demonstrate the benchmarking utility of MultiHopRAG in two experiments. The first experiment compares different embedding models for retrieving evidence for multi-hop queries. In the second experiment, we examine the capabilities of various state-of-the-art LLMs, including GPT-4, PaLM, and Llama2-70B, in reasoning and answering multi-hop queries given the evidence. Both experiments reveal that existing RAG methods perform unsatisfactorily in retrieving and answering multi-hop queries. We hope MultiHop-RAG will be a valuable resource for the community in developing effective RAG systems, thereby facilitating greater adoption of LLMs in practice. The MultiHopRAG and implemented RAG system is publicly available at https://github.com/yixuantt/ MultiHop-RAG/ .

## 1 Introduction

The emergence of large language models (LLMs), such as ChatGPT, has fostered a wide range of innovations, powering intelligent chatbots and other natural language processing (NLP) applications (Ope- nAI, 2023). One promising use case is RetrievalAugmented Generation (RAG) (Asai et al., 2023), which optimizes the output of a large language model by referencing an external knowledge base outside of the LLM training data sources before generating a response. RAG improves LLM's response (Borgeaud et al., 2022) and also mitigates the occurrence of hallucinations, thereby enhancing the models' credibility (Gao et al., 2023). LLMbased frameworks, such as LlamaIndex (Liu, 2022) and LangChain (Chase, 2022), specialize in supporting RAG pipelines.

Figure 1: RAG with multi-hop query.

In real-world Retrieval-Augmented Generation (RAG) applications, a user's query often necessitates retrieving and reasoning over evidence from multiple documents, a process known as multi-hop query . For instance, consider financial analysis using a database of financial reports. A financial analyst might query, Which company among Google, Apple, and Nvidia reported the largest profit margins in their third-quarter reports for 2023? or inquire about a specific company's performance over time, such as How does Apple's sales trend look over the past three years? These queries require evidence from multiple documents to formulate an answer. Due to the multifaceted nature of such queries, involving information from various sources, traditional similarity matching methods like cosine similarity between query and financial report chunk embeddings might not yield optimal results. We demonstrate this multi-hop retrieval process in Figure 1.

Table 1: An example of a multi-hop query, including supporting evidence from two news articles, the paraphrased claim, the bridge-topic and bridge-entity, and the corresponding answer.

| News source                | Fortune Magazine                                                                                                                                                                                                | The Sydney Morning Herald                                                                                                                                                                                       |
|----------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Evidence                   | Back then, just like today, home prices had boomed for years before Fed officials were ultimately forced to hike interest rates aggressively in an attempt to fight inflation.                                  | Postponements of such reports could complicate things for the Fed, which has insisted it will make upcoming decisions on interest rates based on what incoming data say about the economy.                      |
| Claim                      | Federal Reserve officials were forced to aggressively hike interest rates to combat inflation after years of booming home prices.                                                                               | The Federal Reserve has insisted that it will base its upcoming decisions on interest rates on the incoming economic data.                                                                                      |
| Bridge-Topic Bridge-Entity | Interest rate hikes to combat inflation Federal Reserve                                                                                                                                                         | Interest rate decisions based on economic data Federal Reserve                                                                                                                                                  |
| Query                      | Does the article from Fortune suggest that the Federal Reserve's interest rate hikes are a response to past conditions, such as booming home prices, while The Sydney Morning Herald article indicates that the | Does the article from Fortune suggest that the Federal Reserve's interest rate hikes are a response to past conditions, such as booming home prices, while The Sydney Morning Herald article indicates that the |
| Answer                     | Yes                                                                                                                                                                                                             | Yes                                                                                                                                                                                                             |

However, existing RAG benchmarks, such as RGB (Chen et al., 2023) and RECALL (Liu et al., 2023), mainly evaluate a simple case where the answer of a query can be retrieved and solved using one single piece of evidence. None of these benchmarks assess the retrieval and reasoning capability of LLMs for complex multi-hop queries. To address this gap and make RAG benchmarking more closely resemble real-world scenarios, in this paper, we introduce MultiHop-RAG . To our knowledge, MultiHop-RAG is one of the first RAG datasets focusing specifically on multi-hop queries.

Based on the RAG queries commonly encountered in real-world scenarios, we first categorize multi-hop queries into four types: Inference query , Comparison query , Temporal query , and Null query . The first three types - Inference, Comparison, and Temporal - require the retrieval and analysis of evidence from multiple sources, encompassing tasks like inferring relationships, comparing data points, and sequencing events over time. The Null query represents a scenario where the query cannot be derived from the knowledge base. This category is crucial for assessing whether an LLM might hallucinate an answer to a multi-hop query when the retrieved text lacks relevance.

We construct our RAG knowledge base using a collection of news articles. Using GPT-4 as a data generator, we then take an extensive procedure to construct a diverse set of multi-hop queries, each requiring the retrieval and reasoning over multiple documents. An example of query construction is shown in Table 1. First, we begin by extracting factual sentences from each news article as evidence. For example, an extracted piece of evidence from an article may state: 'Back then, just like today, home prices had boomed for years before Fed officials were ultimately forced to hike interest rates aggressively in an attempt to fight inflation.' Second, we input each evidence piece into GPT-4, prompting it to rephrase the evidence into a claim. This claim is clarified with a disambiguated topic and entity. For instance, GPT-4 might rephrase the aforementioned evidence into: 'Federal Reserve officials were forced to aggressively hike interest rates to combat inflation after years of booming home prices', identifying 'Interest rate hikes to combat inflation' as the topic and 'Federal Reserve' as the entity. These topics and entities act as bridges for constructing multi-hop queries, known as bridge-topic or bridge-entity. Next, we use GPT4 to generate specific multi-hop queries related to the same bridge-topic or bridge-entity, accompanied by the correct answers. Lastly, we undertake a validation step to ensure the data quality.

We demonstrate the benchmarking capabilities of MultiHop-RAG using two experiments, utilizing a RAG system implemented with LlamaIndex (Liu, 2022). The first experiment involves a comparison of different embedding models for retrieving relevant evidence for multi-hop queries. In the second experiment, we assess the reasoning and answering abilities of various state-of-the-art LLMs, including GPT-4, GPT-3.5, PaLM, Claude-2, Llama2-70B, and Mixtral-8x7B, for multi-hop queries when retrieved text is provided. The results from both experiments indicate that the current RAG implementations are inadequate for effectively retrieving and answering multi-hop queries. We publicly release this challenging MultiHop-RAG dataset and hope it will be a valuable resource for the community in developing and benchmarking RAG systems, thereby unleashing the great potential of generative AI in practice.

## 2 RAG with multi-Hop queries

## 2.1 Retrieval-augmented Generation (RAG)

In an RAG application, we utilize an external corpus, denoted as D , which comprises multiple documents and serves as the knowledge base. Each document within this corpus, represented as d i ∈ D , is segmented into a set of chunks.These chunks are then transformed into vector representations using an embedding model and stored in an embedding database. Given a user query q , the system typically retrieves the top-K chunks that best match the query. These chunks constitute the retrieval set for query q , represented as R q = { r 1 , r 2 , ..., r K } . The retrieved chunks, combined with the query and an optional prompt, are then fed into an LLM to generate a final answer, following the format: LLM ( q, R q , prompt ) → answer .

## 2.2 Multi-Hop Query

We define a multi-hop query as one that requires retrieving and reasoning over multiple pieces of supporting evidence to provide an answer. In other words, for a multi-hop query q , the chunks in the retrieval set R q collectively provide an answer to q . For example, the query "Which company among Google, Apple, and Nvidia reported the largest profit margins in their third-quarter reports for 2023?" requires 1) retrieving relevant pieces of evidence related to profit margins from the reports of the three companies; 2) generating an answer by comparing and reasoning from the multiple pieces of retrieved evidence. This differs from a singlehop query such as "What is Google's profit margin in the third-quarter reports for 2023," where the answer can be directly derived from a single piece of evidence.

Based on the queries commonly used in realworld RAG systems, we identify four types of multi-hop queries. For each type, we present a hypothetical query within the context of a financial RAG system, where the knowledge base consists of a collection of annual reports.

Inference query: For such a query q , the answer is deduced through reasoning from the retrieval set R q . An example of an inference query might be: Which report discusses the supply chain risk of Apple, the 2019 annual report or the 2020 annual report?

Comparison query: For such a query q , the answer requires a comparison of evidence within the retrieval set R q . For instance, a comparison query might ask: Did Netflix or Google report higher revenue for the year 2023? "

Temporal query: For such a query q , the answer requires an analysis of the temporal information of the retrieved chunks. For example, a temporal query may ask: Did Apple introduce the AirTag tracking device before or after the launch of the 5th generation iPad Pro?

Null query: For such as query q , the answer cannot be derived from the retrieved set R q . We include the null query to assess the generation quality, especially regarding the issue of hallucination. For a null query, even though a retrieved set is provided, an LLM should produce a null response instead of hallucinating an answer. For example, assuming ABCD is a non-existent company, a null query might ask: What are the sales of company ABCD as reported in its 2022 and 2023 annual reports?

## 2.3 Evaluation Metrics

An RAG system handling multi-hop queries can be assessed from two key aspects: retrieval evaluation and generation evaluation.

Retrieval Evaluation: Evidently, the quality of the retrieval set R q determines the final generation quality. We compare the retrieved set with the ground truth evidence associated with each query, except for the null queries, as they have no evidence to derive from. Assuming the topK chunks are retrieved, i.e., |R q | = K , we use retrieval evaluation metrics including Mean Average Precision at K (MAP@K), Mean Reciprocal Rank at K (MRR@K), and Hit Rate at K (Hit@K). MAP@K measures the average top-K retrieval precision across all queries. MRR@K calculates the average of the reciprocal ranks of the first relevant chunk for each query, considering the top-K retrieved set. Hit@K metric measures the fraction of evidence that appears in the top-K retrieved set.

Response Evaluation: Since the multi-hop query requires reasoning over multiple pieces of retrieved chunks, we can also evaluate the reasoning capability of the LLM by comparing the LLM response with the ground truth answer of the query.

Figure 2: MultiHop-RAG Construction Pipeline.

## 3 A Benchmarking Dataset: MultiHop-RAG

In this section, we provide detailed information on the construction of the MultiHop-RAG dataset. Specifically, we describe the process of creating a set of multi-hop queries, along with the corresponding ground truth evidence sets and answers derived from a collection of news articles.

## 3.1 MultiHop-RAG Construction

Step 1: Dataset Collection. We download a news dataset using the mediastack API 1 , a REST API interface delivering worldwide news data. The news data source comprises various English-language websites covering a range of news categories: entertainment, business, sports, technology, health, and science. To mimic real-world RAG scenarios, where the knowledge base data, such as an enterprise's internal data, may differ from the LLMs' training data, we select news articles published from September 26, 2023, to December 26, 2023. This timeframe extends beyond the knowledge cutoff of some widely-used LLMs, including ChatGPT and LLaMA, as of the time of writing. This selection also helps in teasing out the possibility of the underlying LLM having been exposed to these news articles. We only keep articles with a token length greater than or equal to 1,024. Every news article is paired with metadata, including the title, publish date, author, category, URL, and news source.

1 https://mediastack.com/

Step 2: Evidence Extraction. For each article, we extract factual or opinion sentences using a trained language model 2 . These factual sentences are later used as evidence for answering multi-hop queries. We retain only those news articles containing evidence that may have overlapping keywords with other news articles. This allows us to later create multi-hop queries where the answer's evidences are drawn from multiple sources.

Step 3: Claim, Bridge-Entity, Bridge-Topic Generation. Our goal is to use GPT-4 to automatically generate high-quality multi-hop queries using the evidence set. However, the raw evidence obtained from Step 2 is not ideal for query generation due to inconsistency in linguistic structure. For example, some pieces of evidence use pronouns to refer to subjects and lack the actual entity in the text. To address this, we employ GPT-4 to paraphrase the evidence, which we refer to as claims , given the original evidence and its context. To ensure consistency between the generated claim and the evidence, we further perform fact-checking using the UniEval (Zhong et al., 2022) framework to verify the alignment between the evidence and claim. Appendix A presents the prompt used for GPT-4 for claim generation.

Bridge-Entity and Bridge-Topic: The shared entity or topic across pieces of evidence is referred to as the bridge-entity or bridge-topic. These bridgeentities or bridge-topics can be used to link different pieces of evidence from which a multi-hop query's answer is derived. For example, in a claim such as 'Google reports its third-quarter results for 2023, showcasing a detailed overview of its financial performance, including revenue growth, profit margins' , the term profit margin can be viewed as a bridge-topic and the term Google can be viewed as a bridge-entity that links the different pieces of evidence. We prompt GPT-4 to identify the bridgeentity and bridge-topic for each claim. Appendix A also presents the prompt used for GPT-4 for bridge generation.

Step 4: Query and Answer Generation. In this step, we leverage the bridge-entity or bridge-topic to generate multi-hop queries. Specifically, we first group the claims having the same bridge-entity or bridge-topic into a claim set. We restrict the claim set to have at least two claims but no more than four claims. For each type of query, we feed the claim set to GPT-4 and prompt it with an instruction to generate a query with information from each claim. Below, we explain the specifications for different multi-hop query types. In the construction of each query, we also include the source of the news article where the supporting evidence is associated with to mimic real-world RAG scenarios. Appendix A presents the prompts used for GPT-4 for query generation.

2 https://huggingface.co/lighteternal/fact-or-opinion-xlmr-

Inference Query: These queries are formulated by synthesizing the various characterizations of the bridge-entity across multiple claims, with the final answer being the identification of the entity itself.

Comparison Query: These queries are structured to compare the similarities and differences related to the bridge entity or topic. The resultant answer to such queries is typically a definitive 'yes' or 'no', based on the comparison.

Temporal Query: These queries explore the temporal ordering of events across different points in time. The answer to such queries is typically a 'yes' or 'no' or a single temporal indicator word like 'before' or 'after'.

Null Query: Null query is a query whose answer cannot be derived from the retrieved set. To create null queries, we generate multi-hop queries using entities that do not exist in the existing bridgeentities. To add complexity, we also include fictional news source metadata when formulating these questions, ensuring that the questions do not reference any contextually relevant content from the knowledge base. The answer to the null query should be 'insufficient information' or similar.

Step 5: Quality Assurance. Finally, we use two approaches to reassure the dataset quality. First, we manually review a subset sample of the generated multi-hop queries, their corresponding evidence sets, and the final answers. The results of the manual examination indicate a high degree of accuracy and data quality. Second, we utilize GPT-4 to assess each example in the dataset against the following criteria: 1) The generated query must utilize all provided evidence in formulating the response; 2) The query should be answerable solely based on the provided evidence; 3) The response to the generated query should be either a single word or a specific entity; 4) The query must conform to its designated query type.

Table 2: Descriptive statistics of the news article knowledge base in MultiHop-RAG.

| Category      |   Avg. Tokens |   Entry Count |
|---------------|---------------|---------------|
| technology    |        2262.3 |           172 |
| entertainment |        2084.3 |           114 |
| sports        |        2030.6 |           211 |
| science       |        1745.5 |            21 |
| business      |        1723.8 |            81 |
| health        |        1481.1 |            10 |
| total         |        2046.5 |           609 |

Table 3: The distribution of query types in MultiHopRAG.

| Query Category   |   Entry Count | Percentage   |
|------------------|---------------|--------------|
| Inference Query  |           816 | 31.92%       |
| Comparison Query |           856 | 33.49%       |
| Temporal Query   |           583 | 22.81%       |
| Null Query       |           301 | 11.78%       |
| Total            |         2,556 | 100.00%      |

## 3.2 Descriptive Statistics

The MultiHop-RAG dataset contains six different types of news articles, covering 609 distinct news, with an average of 2,046 tokens. The distribution of the news categories is shown in Table 2. MultiHopRAG contains four types of multi-hop queries and the distribution of these queries is shown in Table 3. In total, about 88% of queries in the dataset are non-null queries where answers can be retrieved and reasoned from the knowledge base. In addition, the form of queries exhibits considerable diversity. Approximately 27% of interrogative queries start with "does," around 15% initiate with "what," a similar proportion start "which," and 14% begin with "who," with the remainder incorporating a small percentage of other interrogative words such as "when." Moreover, the number of evidence required to answer a multi-hop query varies. Table 4 shows the distribution of evidence numbers for each query in the dataset. Around 42% of queries can be answered using two pieces of evidence, while approximately 30% and 15% of queries can be answered using three or four pieces of evidence, respectively.

## 4 Benchmarking RAG system using MultiHop-RAG

MultiHop-RAG can be used as a benchmark for various RAG-related tasks. Broadly speaking, RAG- related tasks can be categorized as retrieval-related tasks and generation-related tasks . A retrievalrelated task focuses on retrieving relevant text from the knowledge base, while a generation-related task focuses on generating high-quality responses given the retrieved text. In this section, we showcase two use cases for each task where MultiHop-RAG can be employed.

Table 4: The distribution of the number of evidence required to answer multi-hop queries in MultiHop-RAG.

| Num. of Evidence Needed   |   Count | Percentage   |
|---------------------------|---------|--------------|
| 0 (Null Query)            |     301 | 11.78%       |
| 2                         |    1078 | 42.18%       |
| 3                         |     779 | 30.48%       |
| 4                         |     398 | 15.56%       |
| Total                     |   2,556 | 100.00%      |

## 4.1 Retrieval-related Task

An important design choice in an RAG system is the selection of the embedding model. An embedding model converts data into numerical vectors and subsequently stores these vectors in embedding databases. In this experiment, we evaluate different embedding models by examining their retrieval quality.

Experiment Setup: We implement an RAG system using the LlamaIndex framework (Liu, 2022). We partition the documents in the MultiHop-RAG knowledge base into chunks, each consisting of 256 tokens. We then convert the chunks using an embedding model and save the embeddings into a vector database. Similarly, in the retrieval step, we convert a query using the same embedding model and retrieve the top-K most relevant chunks that have the highest cosine similarity with the query embedding. In this experiment, we test a variety set of embedding models, including the ada-embeddings by OpenAI (text-embedding-ada-002, text-search-adaquery-001), voyage-02 3 , llm-embedder (Zhang et al., 2023), bge-large-en-v1.5 (Xiao et al., 2023), jina-embeddings-v2-base-en (Günther et al., 2023), e5-base-v2 (Wang et al., 2022), and instructor-large (Su et al., 2023). NULL queries are excluded in this experiment because there is no matching evidence to the query. Additionally, we also include a Reranker module to examine the retrieval performance, using bge-reranker-large (Xiao et al., 2023). After retrieving 20 related chunks using the em- bedding model, we further select the top-K chunks using the Reranker.

3 https://www.voyageai.com/

Experiment Result: Table 5 shows the retrieval result of using different embedding models. It shows that there is still a significant gap in retrieving relevant evidence for the multi-hop queries. While Rerank can effectively improve retrieval relevance, the highest Hits@10 is only 0.7467 when the Reranker technique is used. Moreover, the drop in the highest Hits@4 to 0.6625 is worrisome. In practical RAG systems, the underlying LLM often has a context window limit. As a result, the number of retrieved chunks is usually restricted to a small number. The low values of the retrieval metrics highlight the challenges in retrieving relevant pieces of evidence for multi-hop queries when using direct similarity matching between the multihop query and text chunks.

## 4.2 Generation-related Task

The underlying LLMs play a crucial role in generating responses in an RAG system. In this experiment, we evaluate the quality of generated responses under two different settings. In the first setting, we employ the best-performing retrieval model, namely voyage-02 with bge-reranker-large, as indicated in Table 5, to retrieve the top-K texts and then feed them into the LLM. In the second setting, we use the ground-truth evidence associated with each query as the retrieved text for the LLM. This setting represents a ceiling performance for testing the LLM's response capabilities, as it utilizes the actual evidences.

Experiment Setup: In the first experiment, we retrieve top-6 chunks so that the total length of the retrieved text does not exceed 2,048. All queries in MultiHop-RAG are tested in the experiment. In the second experiment, since the null queries do not have associated evidence, we exclude this type of query in the experiment. For the LLMs used in the experiment, we consider state-of-theart commercial models, including GPT-4 (OpenAI, 2023), GPT-3.5, Claude-2 (Anthropic, 2023), and Google-PaLM (Google, 2023). We obtain answers using the provided API of the respective models. We also assess some open-source models, including Mixtral-8x7b-instruct (Jiang et al., 2024) and Llama-2-70b-chat-hf (Touvron et al., 2023).

Experiment Results: Table 6 shows the response accuracy of different LLMs. First, we can see that the response accuracy rate using the retrieved chunks is not satisfactory, with the state-of-theart GPT-4 model achieving only 0.56 accuracy. This is expected, because the retrieval component falls short in retrieving relevant evidences from the knowledge base. Second, even when we provide the LLM with the ground-truth evidences, we can see that the response accuracy is far from being perfect. Open source LLM such as Llama02-70B and Mixtral-8x7B only achieve an accuracy of 0.32 and 0.36 respectively. GPT-4 achieves strong reasoning capability with an accuracy of 0.89, followed by the second-based LLM Google-PaLM with an accuracy of 0.74.

Table 5: Retrieval performance of different embedding models.

| Embedding                  | Without Reranker   | Without Reranker   | Without Reranker   | Without Reranker   | With bge-reranker-large   | With bge-reranker-large   | With bge-reranker-large   | With bge-reranker-large   |
|----------------------------|--------------------|--------------------|--------------------|--------------------|---------------------------|---------------------------|---------------------------|---------------------------|
| Embedding                  | MRR@10             | MAP@10             | Hits@10            | Hits@4             | MRR@10                    | MAP@10                    | Hits@10                   | Hits@4                    |
| text-embedding-ada-002     | 0.4203             | 0.3431             | 0.6381             | 0.504              | 0.5477                    | 0.4625                    | 0.7059                    | 0.6169                    |
| text-search-ada-query-001  | 0.4203             | 0.3431             | 0.6399             | 0.5031             | 0.5483                    | 0.4625                    | 0.7064                    | 0.6174                    |
| llm-embedder               | 0.2558             | 0.1725             | 0.4499             | 0.3189             | 0.425                     | 0.3059                    | 0.5478                    | 0.4756                    |
| bge-large-en-v1.5          | 0.4298             | 0.3423             | 0.6718             | 0.5221             | 0.563                     | 0.4759                    | 0.7183                    | 0.6364                    |
| jina-embeddings-v2-base-en | 0.0621             | 0.031              | 0.1479             | 0.0802             | 0.1412                    | 0.0772                    | 0.1909                    | 0.1639                    |
| intfloat/e5-base-v2        | 0.1843             | 0.1161             | 0.3556             | 0.2334             | 0.3237                    | 0.2165                    | 0.4176                    | 0.3716                    |
| voyage-02                  | 0.3934             | 0.3143             | 0.6506             | 0.4619             | 0.586                     | 0.4795                    | 0.7467                    | 0.6625                    |
| hkunlp/instructor-large    | 0.3458             | 0.265              | 0.5717             | 0.4229             | 0.5115                    | 0.4118                    | 0.659                     | 0.5775                    |

Table 6: Generation accuracy of LLMs.

| Models                | Accuracy        | Accuracy           |
|-----------------------|-----------------|--------------------|
|                       | Retrieved Chunk | Ground-truth Chunk |
| GPT-4                 | 0.56            | 0.89               |
| ChatGPT               | 0.44            | 0.57               |
| Llama-2-70b-chat-hf   | 0.28            | 0.32               |
| Mixtral-8x7B-Instruct | 0.32            | 0.36               |
| Claude-2.1            | 0.52            | 0.56               |
| Google-PaLM           | 0.47            | 0.74               |

Figure 3 shows the detailed results of different query types for GPT-4 and Mixtral-8x7B-instruct. Both models show relatively high robustness on null queries, meaning they are generally good at determining when a query cannot be answered based on the retrieved text. This is encouraging because one benefit of RAG is to mitigating the LLM hallucination issue by augmenting LLM with retrieval knowledge. However, Mixtral-8x7B model performs significantly worse than the GPT-4 in comparison and temporal queries. Upon reviewing the incorrect responses, we find that Mixtral-8x7B fails to accurately handle logical negation, leading to misinterpretation of statements and thus a low performance in the comparison queries. In addition, Mixtral-8x7B often fails to correctly identify the chronological order of events, which is crucial for answering temporal queries where timing is a key factor. Taken together, this experiment demonstrates that there is still room for improvement in the reasoning capabilities of LLMs, particularly those that are open-source, for multi-hop queries.

Figure 3: Generation accuracy for different query types.

## 4.3 Other Use Cases

Beyond embedding models and LLM generation, there are other areas worth exploring. For example, query decomposition is a widely utilized technique in RAG frameworks, such as LLamaIndex. This process involves breaking down the query into smaller segments; it targets a single document for retrieval and integrates the information subsequently, thereby potentially enhancing retrieval accuracy. Another advanced and promising approach involves building LLM-based agents that can automatically plan and execute multi-hop queries, such as AutoGPT (Gravitas, 2023). Another area of interest is the hybrid retrieval approach, which combines keyword and embedding matching tech- niques. We believe that there are many potential areas for enhancing RAG's performance on multihop queries, and the curated dataset MultiHopRAG can be a valuable resource to the community.

## 5 Related Work

RAG Evaluation: As RAG systems gain increasing popularity, a variety of RAG benchmarking datasets and evaluation tools have been developed. For instance, RGB (Chen et al., 2023) and RECALL (Liu et al., 2023) evaluate the performance of LLMs in generating responses for RAG systems under conditions involving noisy, integrative, and counterfactual queries. However, both datasets primarily focus on evaluating the generation aspect of RAG systems without specifically addressing their retrieval accuracy. In addition, recent advancements have been made in automated RAG evaluation tools, such as ARES (Saad-Falcon et al., 2023) and RAGAS (Es et al., 2023). These tools utilize LLMs to automatically assess the quality of RAG generation, yet they do not introduce benchmarking datasets. Our work introduces one of the first RAG benchmarking datasets, consisting of a knowledge base, a large collection of multi-hop queries, their ground-truth answers, and the associated supporting evidence, thereby complementing existing RAG evaluations.

Retrieval datasets: Apart from the context of RAG, several benchmarking datasets exist for information retrieval evaluation. The FEVER (Fact Extraction and VERification) dataset, for instance, contains claims classified as Supported, Refuted, or NotEnoughInfo by the given Wikipedia article (Thorne et al., 2018). Similarly, the SciFact dataset comprises scientific claims paired with evidencecontaining abstracts (Wadden et al., 2020). However, the claims in both datasets are single-hop statements, and the supporting evidence is from one single article, in contrast to the multi-hop queries discussed in this paper. Another dataset, HoVer, involves claims that require extracting and reasoning from multiple Wikipedia articles (Jiang et al., 2020). However, unlike our dataset, HoVer focuses solely on classifying claims as either supported or not supported by the articles without evaluating an LLM generation step. Moreover, in HoVer, the Wikipedia articles from which evidence is drawn are given for claim verification, which is significantly different from our setting, where relevant pieces of evidence need to be extracted from a large knowledge base. Separately, (Kamalloo et al., 2023) evaluates a range of commercial embedding APIs for information retrieval, but this evaluation is not contextualized within the framework of RAG systems either.

Multi-document QA datasets: Questionanswering (QA) is a fundamental task in NLP, and several popular benchmarks, such as HotpotQA (Yang et al., 2018), MultiRC (Khashabi et al., 2018), and 2WikiMultiHopQA (Ho et al., 2020), aim to achieve QA from multiple sources of documents. This task is similar to our multi-hop query RAG task, as both involve reasoning from multiple sources of information. However, these datasets primarily focus on assessing a model's reasoning skills, and they do not emphasize the retrieval of evidence from a knowledge base. Additionally, their primary data sources Wikipedia, significantly overlap with the training data of most existing LLMs. If we use these sources for benchmarking RAG systems, there is a potential concern that LLM responses might rely on training knowledge rather than reasoning from the retrieved knowledge base.

## 6 Conclusion

In this work, we introduce MultiHop-RAG, a novel and unique dataset designed for queries that require retrieval and reasoning from multiple pieces of supporting evidence. These types of multi-hop queries represent user queries commonly encountered in real-world scenarios. MultiHop-RAG consists of a knowledge base, a large collection of multi-hop queries, their ground-truth answers, and the associated supporting evidence. This paper details the creation process of MultiHop-RAG, employing a hybrid approach that integrates human effort with GPT-4. Additionally, we explore two use cases of MultiHop-RAG in the benchmarking of RAG systems, thereby highlighting the potential applications of this dataset. By publicly releasing MultiHop-RAG, we aim to provide a valuable resource to the community, contributing to the advancement and benchmarking of RAG systems.

## Limitations

This work has several limitations that can be improved in future research. First, our ground truth answers are restricted to simple responses such as 'yes", 'no", entity names, or temporal indicators like 'before" or 'after" to facilitate the use of a straightforward accuracy metric for evaluating generation performance. Future work could consider allowing free text as answers and employing more sophisticated metrics to assess generation quality. Second, the current dataset limits supporting evidence for a query to a maximum of four pieces. Future work can extend the dataset by including queries that require retrieving and reasoning from even more evidence. Lastly, while our experiments utilize a basic RAG framework using LlamaIndex, future work could involve evaluating the answering of multi-hop queries using more advanced RAG frameworks or LLM-agent frameworks.

## References

Anthropic. 2023. Claude 2.1 (May version). https: //api.anthropic.com/v1/messages . Claude 2.1.

Akari Asai, Sewon Min, Zexuan Zhong, and Danqi Chen. 2023. Retrieval-based language models and applications. In Proceedings of the 61st Annual Meeting of the Association for Computational Linguistics (Volume 6: Tutorial Abstracts) , pages 41-46.

Sebastian Borgeaud, Arthur Mensch, Jordan Hoffmann, Trevor Cai, Eliza Rutherford, Katie Millican, George Bm Van Den Driessche, Jean-Baptiste Lespiau, Bogdan Damoc, Aidan Clark, Diego De Las Casas, Aurelia Guy, Jacob Menick, Roman Ring, Tom Hennigan, Saffron Huang, Loren Maggiore, Chris Jones, Albin Cassirer, Andy Brock, Michela Paganini, Geoffrey Irving, Oriol Vinyals, Simon Osindero, Karen Simonyan, Jack Rae, Erich Elsen, and Laurent Sifre. 2022. Improving language models by retrieving from trillions of tokens. In Proceedings of the 39th International Conference on Machine Learning , volume 162 of Proceedings of Machine Learning Research , pages 2206-2240. PMLR.

Harrison Chase. 2022. LangChain.

Jiawei Chen, Hongyu Lin, Xianpei Han, and Le Sun. 2023. Benchmarking large language models in retrieval-augmented generation.

Shahul Es, Jithin James, Luis Espinosa-Anke, and Steven Schockaert. 2023. Ragas: Automated evaluation of retrieval augmented generation.

- Tianyu Gao, Howard Yen, Jiatong Yu, and Danqi Chen. 2023. Enabling large language models to generate text with citations.

Google. 2023. PaLM 2 (May version). https://generativelanguage.googleapis. com/v1beta2/models/ . Chat-bison-002.

[Significant Gravitas. 2023. Autogpt. https://github. com/Significant-Gravitas/AutoGPT .](https://github.com/Significant-Gravitas/AutoGPT)

Michael Günther, Jackmin Ong, Isabelle Mohr, Alaeddine Abdessalem, Tanguy Abel, Mohammad Kalim Akram, Susana Guzman, Georgios Mastrapas, Saba Sturua, Bo Wang, Maximilian Werk, Nan Wang, and Han Xiao. 2023. Jina embeddings 2: 8192token general-purpose text embeddings for long documents.

Xanh Ho, Anh-Khoa Duong Nguyen, Saku Sugawara, and Akiko Aizawa. 2020. Constructing a multihop QA dataset for comprehensive evaluation of reasoning steps. In Proceedings of the 28th International Conference on Computational Linguistics , pages 6609-6625, Barcelona, Spain (Online). International Committee on Computational Linguistics.

Albert Q. Jiang, Alexandre Sablayrolles, Antoine Roux, Arthur Mensch, Blanche Savary, Chris Bamford, Devendra Singh Chaplot, Diego de las Casas, Emma Bou Hanna, Florian Bressand, Gianna Lengyel, Guillaume Bour, Guillaume Lample, Lélio Renard Lavaud, Lucile Saulnier, MarieAnne Lachaux, Pierre Stock, Sandeep Subramanian, Sophia Yang, Szymon Antoniak, Teven Le Scao, Théophile Gervet, Thibaut Lavril, Thomas Wang, Timothée Lacroix, and William El Sayed. 2024. Mixtral of experts.

Yichen Jiang, Shikha Bordia, Zheng Zhong, Charles Dognin, Maneesh Singh, and Mohit Bansal. 2020. HoVer: A dataset for many-hop fact extraction and claim verification. In Findings of the Conference on Empirical Methods in Natural Language Processing (EMNLP) .

Ehsan Kamalloo, Xinyu Zhang, Odunayo Ogundepo, Nandan Thakur, David Alfonso-Hermelo, Mehdi Rezagholizadeh, and Jimmy Lin. 2023. Evaluating embedding apis for information retrieval. arXiv preprint arXiv:2305.06300 .

Daniel Khashabi, Snigdha Chaturvedi, Michael Roth, Shyam Upadhyay, and Dan Roth. 2018. Looking Beyond the Surface: A Challenge Set for Reading Comprehension over Multiple Sentences. In Proc. of the Annual Conference of the North American Chapter of the Association for Computational Linguistics (NAACL) .

Jerry Liu. 2022. LlamaIndex.

- Yi Liu, Lianzhe Huang, Shicheng Li, Sishuo Chen, Hao Zhou, Fandong Meng, Jie Zhou, and Xu Sun. 2023. Recall: A benchmark for llms robustness against external counterfactual knowledge.
- OpenAI. 2023. GPT4 (Nov 7 version). https://chat. openai.com/chat . gpt-4-1106-preview.
- Jon Saad-Falcon, Omar Khattab, Christopher Potts, and Matei Zaharia. 2023. Ares: An automated evaluation framework for retrieval-augmented generation systems.

Hongjin Su, Weijia Shi, Jungo Kasai, Yizhong Wang, Yushi Hu, Mari Ostendorf, Wen tau Yih, Noah A. Smith, Luke Zettlemoyer, and Tao Yu. 2023. One embedder, any task: Instruction-finetuned text embeddings.

James Thorne, Andreas Vlachos, Christos Christodoulopoulos, and Arpit Mittal. 2018. Fever: a large-scale dataset for fact extraction and verification.

Hugo Touvron, Louis Martin, Kevin Stone, Peter Albert, Amjad Almahairi, Yasmine Babaei, Nikolay Bashlykov, Soumya Batra, Prajjwal Bhargava, Shruti Bhosale, Dan Bikel, Lukas Blecher, Cristian Canton Ferrer, Moya Chen, Guillem Cucurull, David Esiobu, Jude Fernandes, Jeremy Fu, Wenyin Fu, Brian Fuller, Cynthia Gao, Vedanuj Goswami, Naman Goyal, Anthony Hartshorn, Saghar Hosseini, Rui Hou, Hakan Inan, Marcin Kardas, Viktor Kerkez, Madian Khabsa, Isabel Kloumann, Artem Korenev, Punit Singh Koura, Marie-Anne Lachaux, Thibaut Lavril, Jenya Lee, Diana Liskovich, Yinghai Lu, Yuning Mao, Xavier Martinet, Todor Mihaylov, Pushkar Mishra, Igor Molybog, Yixin Nie, Andrew Poulton, Jeremy Reizenstein, Rashi Rungta, Kalyan Saladi, Alan Schelten, Ruan Silva, Eric Michael Smith, Ranjan Subramanian, Xiaoqing Ellen Tan, Binh Tang, Ross Taylor, Adina Williams, Jian Xiang Kuan, Puxin Xu, Zheng Yan, Iliyan Zarov, Yuchen Zhang, Angela Fan, Melanie Kambadur, Sharan Narang, Aurelien Rodriguez, Robert Stojnic, Sergey Edunov, and Thomas Scialom. 2023. Llama 2: Open foundation and finetuned chat models.

David Wadden, Shanchuan Lin, Kyle Lo, Lucy Lu Wang, Madeleine van Zuylen, Arman Cohan, and Hannaneh Hajishirzi. 2020. Fact or fiction: Verifying scientific claims. In Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing (EMNLP) , pages 7534-7550, Online. Association for Computational Linguistics.

Liang Wang, Nan Yang, Xiaolong Huang, Binxing Jiao, Linjun Yang, Daxin Jiang, Rangan Majumder, and Furu Wei. 2022. Text embeddings by weaklysupervised contrastive pre-training. arXiv preprint arXiv:2212.03533 .

Shitao Xiao, Zheng Liu, Peitian Zhang, and Niklas Muennighoff. 2023. C-pack: Packaged resources to advance general chinese embedding.

Zhilin Yang, Peng Qi, Saizheng Zhang, Yoshua Bengio, William W. Cohen, Ruslan Salakhutdinov, and Christopher D. Manning. 2018. HotpotQA: A dataset for diverse, explainable multi-hop question answering. In Conference on Empirical Methods in Natural Language Processing (EMNLP) .

Peitian Zhang, Shitao Xiao, Zheng Liu, Zhicheng Dou, and Jian-Yun Nie. 2023. Retrieve anything to augment large language models.

Ming Zhong, Yang Liu, Da Yin, Yuning Mao, Yizhu Jiao, Pengfei Liu, Chenguang Zhu, Heng Ji, and Jiawei Han. 2022. Towards a unified multidimensional evaluator for text generation.

## A Appendix A: GPT-4 Prompts Used for Data Generation

We present the prompts used for guiding GPT-4 for data generation. Table 7 shows the prompt used for claim generation, along with the corresponding topics and entities within these claims. Table 8, Table 9, and Table 10 respectively show the prompts used for generating multi-hop queries of the inference, comparison, and temporal types.

## B Appendix B: Dataset Examples

In this appendix, we present an example of each type of multi-hop query included in the MultiHopRAG dataset. These examples are illustrated in the respective tables: Table 12 for Inference Queries, Table 13 for Comparison Queries, Table 14 for Temporal Queries, and Table 15 for Null Queries. Each query is paired with a ground-truth answer for the evaluation of generation accuracy, while multiple pieces of supporting evidence are included for assessing retrieval performance. Additionally, metadata such as the title, source, and publication time of the news articles are provided as references.

A "claim" is a statement or assertion made within a text expressing a belief, opinion, or fact. Given evidence from the original context, please extract one claim and its associated topics.

Note: The claim should not contain ambiguous references, such as 'he',' she,' and' it', and should use complete names. If there are multiple topics, give the most dominant one. The target of the claim (one entity)is the specific individual, group, or organization that the statement or assertion within a text is directed towards or about which it is making a case. The topic of the claim should be a simple phrase representing the claim's central argument concept. If there is no claim, please leave it blank. Please generate a claim based on the given evidence. Don't generate the evidence yourself.

Please give the response following this format:

Evidence: [original context]

Claims: [extract claim]

Claim Target: [target]

Claim Topic: [topic]

Here are examples: &lt;examples&gt; Now, it's your turn. &lt;News&gt;

&lt;evidence&gt;

Table 7: Claim Generation Prompting

A multi-hop question is a query requiring multiple inferential leaps or accessing several pieces of information from different locations or sources to arrive at an answer. The following are news articles' metadata and claims come from the articles. All the claims from the article are related to a similar target. Your task is to generate one multi-hop inference question based on the claims. Here are some instructions:

1. Find the Connection: The connection between claims is &lt;target&gt; , which is how these key pieces of information are related or how they can be combined to form a more complex idea.
2. Formulate the Question: Create a question that cannot be answered by relying on just one of the sentences but instead requires understanding and linking the information from all of the sources. The answer is &lt;target&gt; .
3. Ensure Coherence: Make sure the question flows logically from the combined information and is clear and unambiguous.
4. Use the keywords: &lt;key set&gt;

&lt;examples&gt; Context: &lt;Context&gt;

## &lt;Context&gt;

The above are news articles' metadata and claims come from the articles. All the claims from the articles are related to a similar target. Your task is to generate one comparison question based on all the claims from different sources. This question needs to compare some factual elements of the claims that are explicitly stated to find where they agree or differ. The correct answer to this question is expressed as a comparative adjective, a statement of alignment, a simple yes or no. To generate a comparative question from claims, you need to use the following keywords: &lt;key set&gt;

The Good Comparison Questions: &lt;examples&gt; Your Comparison Question:

Table 9: Comparison Query Generation Prompting

## &lt;Context&gt;

Please create a time-sensitive comparison question using metadata and excerpts from multiple news articles. That is to compare the consistency or sequence of reports on similar topics at multiple different time points. If it is to compare the consistency, please clearly mention the news source and time in the question using &lt;time frame&gt; . If it is to compare sequences of reports, just clearly mention the news source and do not mention the timeline. Utilize the following keywords provided in the &lt;key set&gt; to construct the question. The correct answer should based on the factual excerpts and is only one word.

## &lt;examples&gt;

Your time-sensitive comparison question:

Table 10: Temporal Query Generation Prompting

A multi-hop question is a query requiring multiple inferential leaps or accessing several pieces of information from different locations or sources to arrive at an answer. Considering you have read at least two news articles on &lt;entity&gt; , construct a multi-hop question that incorporates all the news sources. The source of the news should be stated in the question. Also, ensure that the answer to the question is a single word/entity. Do not answer this question directly. Just give me the question:

Query: Which platform is at the center of discussions in articles from Music Business Worldwide, Polygon, and FOX News - Health, concerning the policing of AI-driven voice replication, the debate over "reaction" content, and being the most used app overnight by young people?

Answer:

YouTube

## Evidence List:

Title: Sony Music's artists aren't involved in YouTube's new voice-cloning AI experiment.

Source: Music Business Worldwide

Published Time: 2023-11-23T18:48:48+00:00

Fact: During this period of discussion, YouTube has made a number of positive announcements regarding the biggest issue for any rightsholder regarding AI-driven voice replication of artists: their ability to police it.

Title: YouTube demonetizes popular content creator SSSniperwolf after doxxing accusations

Source: Polygon

Published Time: 2023-10-25T18:18:06+00:00

Fact: The debate over "reaction" content on YouTube has been brewing for years, but a recent incident

between two creators has refueled the urgency of the conversation.

Title: Cell phone shocker as 97% of kids use their device during school hours and beyond, says study Source: FOX News - Health

Published Time: 2023-10-01T09:05:26+00:00

Fact: Overnight phone use was primarily spent engaging with the same media, although YouTube appeared to be the longest-running app because videos were often left playing during the night.

Table 12: The example of inference questions

Query: Did the Cnbc | World Business News Leader report on Nike's net income and the article from The Age on the 10-year Treasury yield both report a decrease in their respective financial metrics? Answer: Yes

## Evidence List:

Title: Nike misses revenue expectations for the first time in two years, beats on earnings and gross margin

Source: Cnbc | World Business News Leader

Published Time: 2023-09-28T20:31:00+00:00

Fact: The company's reported net income for the three-month period that ended August 31 was $1.45 billion, or 94 cents per share, compared with $1.47 billion, or 93 cents per share, a year earlier.

Title: ASX set to open higher as Wall Street rebounds; $A rises

Source: The Age

Published Time: 2023-10-04T21:01:01+00:00

Fact: The yield on the 10-year Treasury, which is the centrepiece of the bond market, pulled back from its highest level since 2007, down to 4.73 per cent from 4.80 per cent late on Tuesday.

Table 13: The example of comparison questions

Query: Was the performance of the Chicago Bears' defense reported as improved by Yardbarker after Sporting News highlighted a sack by the Bears' defense on Joshua Dobbs during the NFL 'Monday Night Football' game?

Answer:

Yes

## Evidence List:

Title: Bears vs. Vikings live score, updates, highlights from NFL 'Monday Night Football' game Source: Sporting News

Published Time: 2023-11-27T23:32:04+00:00

Fact: The Bears answer right back and sack Dobbs, with Sweat and Brisker in there to take him down.

Title: Hottest seat on each NFC team: Buns burning for these four head coaches

Source: Yardbarker

Published Time: 2023-11-30T22:29:33+00:00

Fact: In his second season as HC, the defense has improved, but positive results are hard to come by behind a lackluster offense ranked 19th in yards (323.2) and 21st in points per game (20.2).

Table 14: The example of time-sensitive questions

Query: What is the first letter of the CEO's last name in the news article from Bloomberg on TomTom, and what is the first letter of the city where the company's headquarters is located in the news article from Reuters?

Answer:

Insufficient information.

Table 15: The example of negative rejection questions

# Chunking (문서 분할)

![rag_split](figures/rag_split.png)

- Load 한 문서를 지정한 기준의 덩어리(chunk)로 나누는 작업을 진행한다.

## 나누는 이유
1. **임베딩 모델의 컨텍스트 길이 제한**
    - 대부분의 언어 모델은 한 번에 처리할 수 있는 토큰 수에 제한이 있다. 전체 문서를 통째로 입력하면 이 제한을 초과할 수 있어 처리가 불가능해진다.
2. **검색 정확도 향상**
    - 큰 문서 전체보다는 특정 주제나 내용을 다루는 작은 chunk가 사용자 질문과 더 정확하게 매칭된다. 예를 들어, 100페이지 매뉴얼에서 특정 기능에 대한 질문이 있을 때, 해당 기능을 설명하는 몇 개의 문단만 검색되는 것이 더 효과적이다.
    - 사용자 질문에 대해 문서의 모든 내용이 다 관련있는 것은 아니다. Chunking을 통해 가장 관련성 높은 부분만 선별적으로 활용할 수 있어 답변의 품질이 향상된다.
    - 전체 문서에는 질문과 무관한 내용들이 많이 포함되어 있어 모델이 혼란을 겪을 수 있다. 적절한 크기의 chunk는 이런 노이즈를 줄여준다.
3. **계산 효율성**
    - 벡터 유사도 계산, 임베딩 생성 등의 작업이 작은 chunk 단위로 수행될 때 더 빠르고 효율적이다. 메모리 사용량도 줄일 수 있다.

## 주요 Splitter
- **Splitter**는 문서를 분할(chunking)을 처리해주는 도구들이다. Langchain은 분할 대상, 방법에 따라 다양한 splitter를 제공한다.
- **Splitter 의 목표**
  - 가능한 한 **의미 있는 덩어리를 유지**하면서, **최대 길이(chunk_size)**를 넘지 않도록 나누기.
- https://reference.langchain.com/python/langchain_text_splitters/

### CharacterTextSplitter
가장  기본적인 Text spliter
- 한개의 구분자를 기준으로 분리한다. (default: "\n\n")
    - 분리된 조각이 chunk size 보다 작으면 다음 조각과 합칠 수 있다.
        - 합쳤을때 chuck_size 보다 크면 안 합친다. chuck_size 이내면 합친다.
    - 나누는 기준은 구분자이기 때문에 chunk_size 보다 글자수가 많을 수 있다.
- chunk size: 분리된 문서(chunk) 글자수 이내에서 분리되도록 한다.
    -  구분자를 기준으로 분리한다. 구분자를 기준으로 분리한 문서 조각이 chunk size 보다 크더라도 그대로 유지한다. 즉 chunk_size가 우선이 아니라 **seperator** 가 우선이다.
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - seperator: 구분 문자열을 지정. (default: '\n\n')
- CharacterTextSplitter는 단순 스플리터로 overlap기능을 지원하지는 않는다. 단 seperator가 빈문자열("") 일 경우에는 overlap 기능을 지원한다. overlap이란 각 이전 청크의 뒷부분의 문자열을 앞에 붙여 문맥을 유지하는 것을 말한다.
  
### RecursiveCharacterTextSplitter
- RecursiveCharacterTextSplitter는 **긴 텍스트를 지정된 최대 길이(chunk_size) 이하로 나누는 데 효과적인 텍스트 분할기**(splitter)이다.
- 여러 **구분자(separators)를 순차적으로 적용**하여, 가능한 한 자연스러운 문단/문장/단어 단위로 분할하고, 최종적으로는 크기 제한을 만족시킨다.
- 분할 기준 문자
    1. 두 개의 줄바꿈 문자 ("\n\n")
    2. 한 개의 줄바꿈 문자 ("\n")
    3. 공백 문자 (" ")
    4. 빈 문자열 ("")
- 작동 방식
    1. 먼저 가장 높은 우선순위의 구분자("\n\n")를 기준으로 분리한다.
    2. 분할된 조각 중 **chunk_size를 초과하는 조각**에 대해 다음 우선순위 구분자("\n" → " " → "")로 재귀적으로 재분할한다.
    3. 이 과정을 통해 모든 조각(chunk)이 chunk_size를 초과하지 않도록 만든다.  
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - chunk_overlap: 연속된 청크들 간의 겹치는 문자 수를 설정. 새로운 청크 생성 시 이전 청크의 마지막 부분에서 지정된 수만큼의 문자를 가져와서 새 청크의 앞부분에 포함시켜, 청크 경계에서 문맥의 연속성을 유지한다.
      - 구분자에 의해 청크가 나눠지면 정상적인 분리이므로 overlap이 적용되지 않는다.
      - 정상적 구분자로 나눌 수 없어 chunk_size에 맞춰 잘라진 경우 문맥의 연결성을 위애 overlap을 적용한다.
    - separators(list): 구분자를 지정한다. 지정하면 기본 구분자가 지정한 것으로 변경된다.

#### 메소드
- `split_documents(Iterable[Document]) : List[Document]`
    - Document 목록을 받아 split 처리한다.
- `split_text(str) : List[str]`
    - string text를 받아서 split 처리한다. 

In [29]:
text = """123456789012345678901234567890123456789012345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

In [32]:
! uv pip install langchain-text-splitters

Checked 1 package in 69ms


In [40]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

splitter = CharacterTextSplitter(
    chunk_size=60,
    chunk_overlap=10, # default = 200
    separator=""

)
result = splitter.split_text(text)
print(type(result))
print(type(result[0]))
print(len(result))

<class 'list'>
<class 'str'>
4


In [41]:
for chunk in result:
    print(len(chunk), chunk)
    print("---")

60 123456789012345678901234567890123456789012345678901234567890
---
60 1234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLM
---
60 DEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefg
---
55 이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
---


In [42]:
doc = Document(page_content=text)
result_docs = splitter.split_documents([doc])

In [44]:
print(len(result_docs))
print(type(result_docs[0]))

4
<class 'langchain_core.documents.base.Document'>


In [45]:
for doc in result_docs:
    print(doc)
    print("-----")

page_content='123456789012345678901234567890123456789012345678901234567890'
-----
page_content='1234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLM'
-----
page_content='DEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefg'
-----
page_content='이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
-----


In [56]:
text2 = """1234567890123456789012345678901234567890
12345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRST.UVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ RSTUVWXYZ
abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

In [57]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 50,
    chunk_overlap = 10,
    separators=["\n\n", "\n", r"[\.?!,~]", ' ', ''],
    is_separator_regex=True,
)
result2 = splitter.split_text(text2)


In [54]:
print(splitter._separators)

['\n\n', '\n', '[\\.?!,~]', ' ', '']


In [58]:
for txt in result2:
    print(len(txt), txt)
    print("-----")


40 1234567890123456789012345678901234567890
-----
29 12345678901234567890123456789
-----
46 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRST
-----
7 .UVWXYZ
-----
26 가나다라마바사아자차카타파하

아야어여오요우유으이
-----
43 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ
-----
9 RSTUVWXYZ
-----
49 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVW
-----
13 NOPQRSTUVWXYZ
-----


In [66]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
path = "data/olympic.txt"
loader = TextLoader(path, encoding="UTF-8")
docs = loader.load()
print("로드한 문서개수", len(docs))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", r"[\.?!,~]", ' ', ''],
    is_separator_regex=True,
)
split_docs = splitter.split_documents(docs)
print("스플릿한 문서 개수", len(split_docs))

로드한 문서개수 1
스플릿한 문서 개수 61


In [68]:
split_docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

In [70]:
# 문서 로드와 스플릿을 같이 할 수 있다

docs = loader.load_and_split(splitter)
print(len(docs))

61


In [71]:
docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

## Token 수 기준으로 나누기

- LLM 언어 모델들은 입력 토큰 수 제한이 있어서 요청시 제한 토큰수 이상의 프롬프트는 전송할 수 없다.
- 따라서 텍스트를 chunk로 분할할 때는 글자수 보다 **토큰 수를 기준으로 크기를 지정하는 것**이 좋다.  
- 토큰기반 분할은 텍스트의 의미를 유지하면서 분할하는 방식이므로 문자 기반 분할과 같이 단어가 중간잘리는 것들을 방지할 수 있다. 
- 토큰 수 계산할 때는 사용하는 언어 모델에 사용된 것과 동일한 tokenizer를 사용하는 것이 좋다.
  - 예를 들어 OpenAI의 GPT 모델을 사용할 경우 tiktoken 라이브러리를 활용하여 토큰 수를 정확하게 계산할 수 있다.

### [tiktoken](https://github.com/openai/tiktoken) tokenizer 기반 분할
- OpenAI에서 GPT 모델을 학습할 때 사용한 `BPE` 방식의 tokenizer. **OpenAI 언어모델을 사용할 경우 이것을 사용하는 것이 좀 더 정확하게  토큰을 계산할 수 있다.**
- Splitter.from_tiktoken_encoder() 메소드를 이용해 생성
  - `RecursiveCharacterTextSplitter.from_tiktoken_encoder()`
  - `CharacterTextSplitter.from_tiktoken_encoder()`
- 파라미터
  - encode_name: 인코딩 방식(토큰화 규칙)을 지정. OpenAI는 GPT 모델들 마다 다른 방식을 사용했다. 그래서 사용하려는 모델에 맞는 인코딩 방식을 지정해야 한다.
    - `o200k_base`: GPT-4 이후 모델들이 사용한 방식
    - `cl100k_base`: 초기 GPT-4 및 GPT-3.5-Turbo 모델에서 사용된 방식.
    - `r50k_base:` GPT-3 모델에서 사용된 방식 
  - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- tiktoken 설치
  - `pip install tiktoken`

### HuggingFace Tokenizer
- HuggingFace 모델을 사용할 경우 그 모델이 사용한 tokenizer를 이용해 토큰 기반으로 분할 한다.
  - 다른 tokenizer를 이용해 분할 할 경우 토큰 수 계산이 다르게 될 수있다.
- `from_huggingface_tokenizer()` 메소드를 이용.
  - 파라미터
    - tokenizer: HuggingFace tokenizer 객체
    - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- `transformers` 라이브러리를 설치해야 한다.
  - `pip install transformers` 

In [72]:
# ! uv pip install tiktoken

Checked 1 package in 637ms


In [78]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"
loader = TextLoader(path, encoding="utf-8")
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    # model_name="gpt-5",
    encoding_name="o200k_base",
    chunk_size=500,
    chunk_overlap=50,

)
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    # model_name="gpt-5",
    encoding_name="o200k_base",
    chunk_size=500,
    chunk_overlap=50,

)
docs = loader.load_and_split(splitter)
print(len(docs))

Created a chunk of size 1027, which is longer than the specified 500
Created a chunk of size 981, which is longer than the specified 500
Created a chunk of size 881, which is longer than the specified 500
Created a chunk of size 608, which is longer than the specified 500
Created a chunk of size 703, which is longer than the specified 500
Created a chunk of size 843, which is longer than the specified 500
Created a chunk of size 925, which is longer than the specified 500
Created a chunk of size 661, which is longer than the specified 500
Created a chunk of size 1305, which is longer than the specified 500
Created a chunk of size 1052, which is longer than the specified 500


18


In [79]:
docs[0].page_content

'올림픽\n올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.\n또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스페셜 올림픽, 데플림픽, 10대 선수들이 참여하는 유스 올림픽 등을 들 수 있다. 그 뿐만 아니라 IOC는 20세기의 변화하는 경제

In [86]:
from transformers import AutoTokenizer
model_id = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
splitter_hf = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(    
    tokenizer=tokenizer,
    chunk_size=500,    
    chunk_overlap=50,
)


In [87]:
docs = loader.load_and_split(splitter_hf)
len(docs)

35

## MarkdownHeaderTextSplitter
- Markdown Header 기준으로 Splitter
- Loading한 문서가 Markdown 문서이고 Header를 기준으로 문서의 내용이 나눠질때 사용.
- https://reference.langchain.com/python/langchain_text_splitters/#langchain_text_splitters.MarkdownTextSplitter

In [94]:
text = """
# 대주제1
- 동물

## 중주제1
- 포유류

- 조류

### 소주제1
- 개
- 고양이
- 까치
- 독수리

# 대주제2
## 중주제2
- 기차
- 배
"""

In [97]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
header_to_split = [
    ("#", "header1"),
    ("##", "header2")
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=header_to_split,
    strip_headers=False,

)
docs = splitter.split_text(text)
len(docs)

3

In [98]:
docs

[Document(metadata={'header1': '대주제1'}, page_content='# 대주제1\n- 동물'),
 Document(metadata={'header1': '대주제1', 'header2': '중주제1'}, page_content='## 중주제1\n- 포유류  \n- 조류  \n### 소주제1\n- 개\n- 고양이\n- 까치\n- 독수리'),
 Document(metadata={'header1': '대주제2', 'header2': '중주제2'}, page_content='# 대주제2  \n## 중주제2\n- 기차\n- 배')]

In [108]:
# olympic.md
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
path = "data/olympic_wiki.md"
loader = TextLoader(path, encoding="utf-8")
header_to_split =[
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=header_to_split
)

In [110]:
docs = loader.load()
doc_txt = "\n".join(doc.page_content for doc in docs)
split_docs = splitter.split_text(doc_txt)
len(split_docs)

25

In [115]:
split_docs[0].metadata

{'h1': '올림픽'}

In [114]:
print(split_docs[0].page_content)

올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  
또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스페셜 올림픽, 데플림픽, 10대 선수들이 참여하는 유스 올림픽 등을 들 수 있다. 그 뿐만 아니라 IOC는 20세기의 변화하는 경제, 정치,